# FLIR thermal training — vehicles and people
Select **Runtime → Change runtime type → GPU**, then **Run all**.
Upload `thermal-vehicles-colab.tar` and its `.sha256` file to **My Drive / ThermalFusion** first.
The notebook copies the archive to the VM and saves checkpoints to Drive after each epoch.
Reopen this notebook and Run all after a runtime disconnect to resume the last saved epoch.
Six classes: person, car, truck, bus, motorcycle, bicycle. **No animal training**: current labels are insufficient.
The held-out test set is not included. FLIR validation is not validation on the Lepton camera.


In [ ]:
%pip -q install ultralytics==8.3.200


In [ ]:
from google.colab import drive
from pathlib import Path
import os, json, shutil, hashlib, tarfile, time
import torch, yaml
from ultralytics import YOLO, settings
assert torch.cuda.is_available(), 'Select a GPU runtime before continuing.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/ThermalFusion')
RUN_NAME = 'flir-vehicles-v1'  # Change this to start a separate experiment.
SAVED = DRIVE / RUN_NAME
SAVED.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/thermal-fusion')
WORK.mkdir(exist_ok=True)
settings.update({'wandb': False, 'mlflow': False, 'comet': False, 'clearml': False, 'neptune': False, 'sync': False})


In [ ]:
archive_name = 'thermal-vehicles-colab.tar'
source = DRIVE / archive_name
assert source.is_file(), f'Upload {archive_name} to {DRIVE}'
expected = (DRIVE / (archive_name + '.sha256')).read_text().split()[0]
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()
local = WORK / archive_name
marker = WORK / '.dataset-sha256'
if not marker.exists() or marker.read_text() != expected:
    print('Copying dataset from Drive...')
    shutil.copyfile(source, local)
    assert sha256(local) == expected, 'Archive checksum mismatch; upload again.'
    with tarfile.open(local) as archive:
        for member in archive.getmembers():
            target = (WORK / member.name).resolve()
            assert target.is_relative_to(WORK.resolve()), 'Unsafe archive path'
            assert member.isfile() or member.isdir(), 'Unsupported archive entry'
        archive.extractall(WORK)
    marker.write_text(expected)
DATA = WORK / 'FLIR_thermal_vehicles_yolo'
NAMES = ['person', 'car', 'truck', 'bus', 'motorcycle', 'bicycle']
config = yaml.safe_load((DATA / 'data.yaml').read_text())
assert config['names'] == NAMES
config['path'] = str(DATA)
(DATA / 'data.yaml').write_text(yaml.safe_dump(config, sort_keys=False))
for split, count in [('train', 10742), ('val', 1144)]:
    images = {p.stem for p in (DATA / 'images' / split).glob('*.png')}
    labels = {p.stem for p in (DATA / 'labels' / split).glob('*.txt')}
    assert len(images) == count and images == labels, f'Incomplete {split} split'
WEIGHTS = WORK / 'models/yolov8n.pt'
assert sha256(WEIGHTS) == 'f59b3d833e2ff32e194b5bb8e08d211dc7c5bdf144b90d2c8412c47ccfc83b36'
print('Dataset ready:', len(NAMES), 'classes')


In [ ]:
# Defaults match the local baseline's input size. Cloud GPU permits a larger batch.
EPOCHS = 30
IMGSZ = 320
BATCH = 16
RESUME = True

def atomic_copy(src, dst):
    dst = Path(dst)
    dst.parent.mkdir(parents=True, exist_ok=True)
    temp = dst.with_name(dst.name + '.tmp')
    shutil.copyfile(src, temp)
    os.replace(temp, dst)

def persist(trainer):
    # Failure to save is surfaced rather than silently losing progress.
    for name in ['last.pt', 'best.pt']:
        src = Path(trainer.save_dir) / 'weights' / name
        if src.exists():
            atomic_copy(src, SAVED / 'weights' / name)
    for name in ['results.csv', 'args.yaml']:
        src = Path(trainer.save_dir) / name
        if src.exists():
            atomic_copy(src, SAVED / name)
    status = {'completed_epochs': trainer.epoch + 1, 'planned_epochs': trainer.epochs,
              'saved_at_unix': time.time(), 'classes': NAMES, 'dataset_sha256': expected,
              'gpu': torch.cuda.get_device_name(0), 'scope': 'FLIR person/vehicles; not validated on Lepton'}
    temp = SAVED / 'progress.json.tmp'
    temp.write_text(json.dumps(status, indent=2))
    os.replace(temp, SAVED / 'progress.json')
    print(f'Checkpoint saved to Drive: epoch {trainer.epoch + 1}/{trainer.epochs}')

last = SAVED / 'weights/last.pt'
if (SAVED / 'COMPLETED.json').exists():
    print('This run has completed. Change RUN_NAME to train another model.')
else:
    if RESUME and last.exists():
        progress = json.loads((SAVED / 'progress.json').read_text())
        assert progress['dataset_sha256'] == expected, 'Dataset differs from saved run'
        checkpoint = WORK / 'resume.pt'
        shutil.copyfile(last, checkpoint)
        model = YOLO(str(checkpoint))
    else:
        assert not last.exists(), 'Use a new RUN_NAME to avoid overwriting a saved run'
        model = YOLO(str(WEIGHTS))
    model.add_callback('on_model_save', persist)
    if RESUME and last.exists():
        model.train(resume=True, device=0, workers=2)
    else:
        model.train(data=str(DATA / 'data.yaml'), epochs=EPOCHS, imgsz=IMGSZ,
                    batch=BATCH, device=0, workers=2, cache=False, amp=True,
                    project=str(WORK / 'runs'), name=RUN_NAME, exist_ok=True,
                    optimizer='AdamW', lr0=0.001, patience=8, seed=20260908,
                    deterministic=True, hsv_h=0.0, hsv_s=0.0, hsv_v=0.15,
                    mosaic=0.5, close_mosaic=5, mixup=0.0, fliplr=0.5,
                    flipud=0.0, plots=True, save=True)
    persist(model.trainer)
    for plot in Path(model.trainer.save_dir).glob('*.png'):
        atomic_copy(plot, SAVED / plot.name)
    (SAVED / 'COMPLETED.json').write_text(json.dumps({'finished_at_unix': time.time()}))
    print('Finished. Best model:', SAVED / 'weights/best.pt')


## After training
Download `ThermalFusion/flir-vehicles-v1/weights/best.pt`, `results.csv`, and `args.yaml` from Drive.
Check per-class validation metrics before integrating into the live viewer.
Cloud runtime duration and GPU availability vary; an interrupted epoch restarts from the previous saved checkpoint.
If the initial batch runs out of GPU memory, reduce BATCH to 8 before the first saved epoch.
Do not close/disconnect the runtime intentionally while training.
